In [1]:
import tensorflow
import tensorflow.keras
import tensorflow.keras.models

I0000 00:00:1784302074.408706 1595053 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1784302077.360776 1595053 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1784302095.935020 1595053 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [3]:
from tensorflow.keras.models import load_model

model = load_model("mnist_model.keras")

In [4]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten (Flatten)               │ (None, 784)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       100,480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 305,312 (1.16 MB)

 Trainable params: 101,770 (397.54 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 203,542 (795.09 KB)

In [5]:
import cv2
import numpy as np
from tensorflow.keras.models import load_model

# --------------------------
# Load Model
# --------------------------
model = load_model("mnist_model.keras")

print("Model Loaded Successfully!")

# --------------------------
# Webcam
# --------------------------
cap = cv2.VideoCapture(1)

prediction = ""
confidence = 0

while True:

    ret, frame = cap.read()

    if not ret:
        break

    frame = cv2.flip(frame, 1)

    # ROI Coordinates
    x1, y1 = 150, 100
    x2, y2 = 430, 380

    cv2.rectangle(frame, (x1, y1), (x2, y2), (0,255,0), 2)

    cv2.putText(
        frame,
        "Put Digit Inside Box",
        (120,80),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (0,255,0),
        2
    )

    if prediction != "":
        cv2.putText(
            frame,
            f"Prediction : {prediction}",
            (120,430),
            cv2.FONT_HERSHEY_SIMPLEX,
            1,
            (0,0,255),
            2
        )

        cv2.putText(
            frame,
            f"Confidence : {confidence:.2f}",
            (120,470),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            (255,0,0),
            2
        )

    cv2.imshow("Digit Recognizer", frame)

    key = cv2.waitKey(1)

    if key == ord('c'):

        roi = frame[y1:y2, x1:x2]

        #################################################
        # PREPROCESSING
        #################################################

        gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)

        gray = cv2.GaussianBlur(gray, (5,5), 0)

        _, thresh = cv2.threshold(
            gray,
            0,
            255,
            cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU
        )

        contours, _ = cv2.findContours(
            thresh,
            cv2.RETR_EXTERNAL,
            cv2.CHAIN_APPROX_SIMPLE
        )

        if len(contours) == 0:
            print("No Digit Found!")
            continue

        largest = max(contours, key=cv2.contourArea)

        x, y, w, h = cv2.boundingRect(largest)

        digit = thresh[y:y+h, x:x+w]

        # Add small padding
        digit = cv2.copyMakeBorder(
            digit,
            20,
            20,
            20,
            20,
            cv2.BORDER_CONSTANT,
            value=0
        )

        digit = cv2.resize(digit, (28,28))

        cv2.imshow(
            "Processed 28x28",
            cv2.resize(digit, (280,280), interpolation=cv2.INTER_NEAREST)
        )

        img = digit.astype(np.float32) / 255.0

        # Model expects (1,28,28)
        img = img.reshape(1,28,28)

        #################################################
        # Prediction
        #################################################

        pred = model.predict(img, verbose=0)

        prediction = np.argmax(pred)

        confidence = np.max(pred)

        print("-----------------------------------")
        print("Prediction :", prediction)
        print("Confidence :", confidence)
        print("-----------------------------------")

    elif key == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

Model Loaded Successfully!


[ WARN:0@39.694] global cap_v4l.cpp:914 open VIDEOIO(V4L2:/dev/video1): can't open camera by index
QFontDatabase: Cannot find font directory /run/media/darshchouhan/New Volume/DL-Algorithm/venv/lib64/python3.12/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /run/media/darshchouhan/New Volume/DL-Algorithm/venv/lib64/python3.12/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /run/media/darshchouhan/New Volume/DL-Algorithm/venv/lib64/python3.12/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /run/media/darshchouhan/New Volume/DL-Algorithm/venv/lib64/python3.

-----------------------------------
Prediction : 2
Confidence : 1.0
-----------------------------------
-----------------------------------
Prediction : 5
Confidence : 1.0
-----------------------------------
-----------------------------------
Prediction : 3
Confidence : 1.0
-----------------------------------
-----------------------------------
Prediction : 7
Confidence : 0.99819726
-----------------------------------
-----------------------------------
Prediction : 7
Confidence : 1.0
-----------------------------------
-----------------------------------
Prediction : 4
Confidence : 1.0
-----------------------------------
-----------------------------------
Prediction : 2
Confidence : 1.0
-----------------------------------
-----------------------------------
Prediction : 2
Confidence : 0.97921866
-----------------------------------
-----------------------------------
Prediction : 0
Confidence : 1.0
-----------------------------------
-----------------------------------
Prediction : 8

ioctl(VIDIOC_QBUF): Bad file descriptor
